# Public UV measurement quality check
Run from the repository root. This optional inspection notebook uses only the standard library; it adds no runtime dependency.
Grain: one station/product/UTC timestamp. Intended use: an exploratory comparison for 5 September 2026, not calibrated forecast qualification. Source: https://uv-data.i-med.ac.at/public/.
The API identifies measurements and units, but supplies neither averaging bounds nor per-value QC/calibration status. Never infer `qc_good=True` from numerical completeness.

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime, timezone
measurements = json.loads(Path('work/observations_20260905.json').read_text())
sites = json.loads(Path('work/observation_sites.json').read_text())
assert len({s['name'] for s in sites}) == len(sites), 'duplicate site key'
start = datetime(2026, 9, 5, tzinfo=timezone.utc)
stop = datetime(2026, 9, 6, tzinfo=timezone.utc)
profile = {}
for s in sites:
    if not (5.3 <= s['longitude'] <= 11.2 and 45.2 <= s['latitude'] <= 48.4):
        continue
    d = measurements.get(s['name'], {}).get('uve')
    if d is None:
        profile[s['name']] = {'missing_product': True}
        continue
    ts = [datetime.fromisoformat(t).astimezone(timezone.utc) for t in d['ts']]
    v = d['measurement']
    assert len(ts) == len(v), 'time/value length mismatch'
    invalid = sum(x is None or not isinstance(x, (int, float)) or not math.isfinite(x) or x < 0 for x in v)
    midday = {t for t in ts if 8 <= t.hour < 16 and t.minute in (15, 45) and start <= t < stop}
    profile[s['name']] = {
        'rows': len(ts), 'unit': d.get('unit'), 'duplicates': len(ts)-len(set(ts)),
        'invalid_values': invalid, 'invalid_fraction': invalid/max(1,len(v)),
        'outside_requested_day': sum(not start <= t < stop for t in ts),
        'strictly_increasing': all(a < b for a,b in zip(ts,ts[1:])),
        'midday_slots_present': len(midday), 'midday_slots_expected': 16,
        'midday_coverage_fraction': len(midday)/16,
        'first_utc': ts[0].isoformat() if ts else None,
        'last_utc': ts[-1].isoformat() if ts else None,
        'maximum_reported_uvi': max(v) if not invalid and v else None}
Path('observation_quality.json').write_text(json.dumps(profile, indent=2)+'\n')
print(json.dumps(profile, indent=2))

## Interpretation and limits
Davos, Weissfluhjoch, Zugspitze and Aosta have all 16 expected midday slots; Dornbirn has none. Missing daytime values must not become zeros or interpolated observations. Treat the Dornbirn series as unsuitable for this comparison (high severity, high confidence).
Missing averaging-period and QC/calibration metadata prevents qualification of hourly forecast accuracy (high severity, high confidence about the missing metadata; cause unknown). Compare paired :15/:45 values only as an explicitly approximate hourly mean. Obtain interval semantics and calibrated/QC-passed data from the network owner before qualification.
This is one retrieved day, not a temporal-drift or multi-season assessment. No source is assumed current merely because it responds successfully. Published site coordinates may be rounded; the comparison records nearest-grid distance and altitude mismatch.
Machine-checkable guards live in `icon_uv/compare_public_uv.py`: units, unique timestamps, finite nonnegative paired values, domain and distance. These do not certify measurement quality. Source URLs, retrieval time and hashes are in `work/observations_manifest.json`.